# 1 - Exploring an unlabelled customer base

This project is **unsupervised end to end**. `D.load_customers()` returns the 9 feature columns and nothing else - the raw CSV's legacy sales-team column is never read, because a real market-entry segmentation would not have one.

Every computation lives in `src/`; this notebook narrates and displays. **Run `python -m src.run_all` first.**

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, joblib
from IPython.display import Image, display
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from src import config as C, data as D, features as F, viz
viz.set_style()

## Load

The clustering population is de-duplicated on the 9 features: identical customers would otherwise drag a prototype toward whatever the export happens to repeat. The prospect file is *not* de-duplicated - every prospect needs an output row.

In [ ]:
raw = D.load_customers_raw()
customers = D.load_customers()
prospects = D.load_prospects()
print("existing market:", customers.shape, "| prospects:", prospects.shape)
print("columns:", list(customers.columns))
customers.head()

In [ ]:
import json
print(json.dumps(D.duplicate_report(raw), indent=1))

## Integrity audit

`validate_schema` also fails loudly if a non-feature column ever leaks into the frame - that is the guard that keeps this project unsupervised.

In [ ]:
print(json.dumps(D.audit(customers), indent=1))

## What each feature can contribute

`concentration` is the share held by the single most common value. A feature near 1.0 splits the population barely at all and cannot do much work in a distance calculation.

In [ ]:
from src import eda
eda.feature_summary(customers)

## Redundancy between features

This matters more without a label than with one. Two strongly associated features effectively vote twice in the distance function, quietly reweighting the segmentation toward whatever they happen to share.

In [ ]:
eda.redundancy_report(customers).head(10)

## Missingness

With no label there is no way to ask whether missingness is *predictive*. The answerable question is whether the gaps co-occur - which would point at one upstream cause.

In [ ]:
m = D.missingness_report(customers)
print("rows missing at least one field: %.1f%%" % m.attrs["pct_rows_with_any_missing"])
m

## Do the new-market prospects resemble the existing market?

With no labels anywhere, this is the only available check on whether segments learned in one market can be applied to the other.

In [ ]:
eda.population_shift(customers, prospects)

## Figures

In [ ]:
for name in ["01_feature_overview", "02_missingness",
             "03_association_matrix", "10_population_shift"]:
    display(Image(filename=str(C.FIGURES / (name + ".png"))))